In [2]:
import pandas as pd
import duckdb
import requests
from datetime import datetime, timedelta, timezone

In [8]:
conn = duckdb.connect("../wynne_strava.db")

In [4]:
conn.sql("SELECT * FROM wynne_activities ORDER BY start_date DESC LIMIT 5")

┌────────────────┬───────────────┬──────────┬─────────────┬──────────────┬──────────────────────┬─────────┬────────────┬──────────────┬─────────────┬─────────────┬──────────────────────┬──────────────────────┬─────────────────────────────┬────────────┬───────────────┬────────────────┬──────────────────┬───────────────────┬─────────────┬───────────────┬───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┬────────────┬─────────┬─────────┬─────────────────────────┬─────────────────────────┬───────────────┬───────────┬───────────────┬───────────────────┬───────────────────────────────┬───────────┬──────────┬─────────────┬───────────────┬────────────────────────────────────────────────────────────┬───────────────────┬──────────┬───────────────────┬────────────┬────────────┬────────────────────────┬──────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [36]:
credentials = {
    "client_id": "230098",
    "client_secret": "57ff9f08daffc8d77771e03bae7c380eed00d7b0",
    "refresh_token": "5d06f94ff6551fe71a0c8ec609a90b152f0997f9",
}

In [37]:
def connect_to_strava(credentials: dict) -> dict:
    """Exchange a refresh token for an access token and return auth headers."""
    response = requests.post(
        "https://www.strava.com/oauth/token",
        data={**credentials, "grant_type": "refresh_token"},
    )
    response.raise_for_status()
    access_token = response.json()["access_token"]
    return {"Authorization": f"Bearer {access_token}"}

In [38]:
def fetch_recent_activities(headers, lookback_days=1):
    now = datetime.now(timezone.utc)
    after = now.replace(hour=0, minute=0, second=0, microsecond=0) - timedelta(days=lookback_days)

    response = requests.get(
        "https://www.strava.com/api/v3/athlete/activities",
        headers=headers,
        params={
            "after": int(after.timestamp()),
            "before": int(now.timestamp()),
            "per_page": 200,
        },
    )
    response.raise_for_status()
    return pd.json_normalize(response.json())

In [39]:
def generate_recent_activities_df(lookback_days=1):
    headers = connect_to_strava(credentials)
    return fetch_recent_activities(headers, lookback_days=lookback_days)

In [40]:
recent_activities_df = generate_recent_activities_df(lookback_days=1)

In [41]:
recent_activities_df

,resource_state,name,distance,moving_time,elapsed_time,total_elevation_gain,type,sport_type,workout_type,device_name,...,external_id,from_accepted_tag,pr_count,total_photo_count,has_kudoed,athlete.id,athlete.resource_state,map.id,map.summary_polyline,map.resource_state
0,2,Afternoon Run,5246.3,1752,1752,2.4,Run,Run,None,Strava App,...,stripped_697DBB00-1A70-49DB-B1A0-D921FD040AEC-...,False,0,0,False,65590982,1,a18283606376,i}{fErunqQCA@Ed@YHKj@e@N]PUHYDE@ONc@Fe@Ni@Ba@F...,2


In [42]:
def insert_recent_activities():

    input_df = generate_recent_activities_df()

    if input_df.empty:
        print("No new activities to insert.")
        return
    
    DB_PATH = "../wynne_strava.db"
    conn = duckdb.connect(DB_PATH)

    # Filter out IDs that already exist
    existing_ids = conn.sql("SELECT id FROM wynne_activities").fetchdf()["id"]
    input_df = input_df[~input_df["id"].isin(existing_ids)]
    
    conn.sql("INSERT INTO wynne_activities BY NAME SELECT * FROM input_df")
    conn.close()

In [46]:
insert_recent_activities()

In [9]:
conn.sql("SELECT * FROM wynne_activitieS ORDER BY start_date_local DESC LIMIT 5")

┌────────────────┬───────────────┬──────────┬─────────────┬──────────────┬──────────────────────┬─────────┬────────────┬──────────────┬─────────────┬─────────────┬──────────────────────┬──────────────────────┬─────────────────────────────┬────────────┬───────────────┬────────────────┬──────────────────┬───────────────────┬─────────────┬───────────────┬───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┬────────────┬─────────┬─────────┬─────────────────────────┬─────────────────────────┬───────────────┬───────────┬───────────────┬───────────────────┬───────────────────────────────┬───────────┬──────────┬─────────────┬───────────────┬────────────────────────────────────────────────────────────┬───────────────────┬──────────┬───────────────────┬────────────┬────────────┬────────────────────────┬──────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [47]:
conn.sql("SELECT * FROM wynne_activitieS ORDER BY start_date_local DESC LIMIT 5")

┌────────────────┬───────────────┬──────────┬─────────────┬──────────────┬──────────────────────┬─────────┬────────────┬──────────────┬─────────────┬─────────────┬──────────────────────┬──────────────────────┬─────────────────────────────┬────────────┬───────────────┬────────────────┬──────────────────┬───────────────────┬─────────────┬───────────────┬───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┬────────────┬─────────┬─────────┬─────────────────────────┬─────────────────────────┬───────────────┬───────────┬───────────────┬───────────────────┬───────────────────────────────┬───────────┬──────────┬─────────────┬───────────────┬────────────────────────────────────────────────────────────┬───────────────────┬──────────┬───────────────────┬────────────┬────────────┬────────────────────────┬──────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [5]:
conn.sql("DELETE FROM wynne_activities WHERE id = 18283606376")

In [ ]:
headers = connect_to_strava(credentials)
recent_activities_df = generate_recent_activities_df()
insert_recent_activities()

In [10]:
conn.close()